# Qwen3-32B Benchmark: vLLM vs llm-d (MaaS)

Confronto tra:
- vLLM diretto su OpenShift AI
- llm-d esposto tramite MaaS

⚠️ Non inserire token o API key nel notebook.
Usare sempre variabili d'ambiente.


In [ ]:
import os

VLLM_DIRECT = "https://qwen3-32b-vllm-agent.apps.privateailab.acic.accenture/v1/chat/completions"
LLMD_ENDPOINT = "https://maas.apps.privateailab.acic.accenture/agent/qwen3-32b/v1/chat/completions"

MODEL_VLLM = "qwen3-32b-vllm"
MODEL_LLMD = "qwen3-32b"

TOKEN_VLLM = os.environ["TOKEN_VLLM"]
TOKEN_LLMD = os.environ["TOKEN_LLMD"]


In [ ]:
import requests
import time
import statistics
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt


In [ ]:
SHORT_PROMPT = "Explain OpenShift AI in 5 bullet points."

LONG_PROMPT = '''
Explain confidential computing on OpenShift.
Describe attestation, Intel TDX, AMD SEV, Kata Containers,
Peer Pods, KBS, Trustee, GPU acceleration, model serving,
RAG architectures and security implications.
''' * 40


In [ ]:
def endpoint_config(target):

    if target == "vllm":
        return {
            "url": VLLM_DIRECT,
            "model": MODEL_VLLM,
            "headers": {
                "Authorization": f"Bearer {TOKEN_VLLM}",
                "Content-Type": "application/json"
            }
        }

    return {
        "url": LLMD_ENDPOINT,
        "model": MODEL_LLMD,
        "headers": {
            "Authorization": f"Bearer {TOKEN_LLMD}",
            "Content-Type": "application/json"
        }
    }


In [ ]:
def send_request(target, prompt, max_tokens=256):

    cfg = endpoint_config(target)

    payload = {
        "model": cfg["model"],
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": max_tokens,
        "stream": False
    }

    start = time.time()

    r = requests.post(
        cfg["url"],
        headers=cfg["headers"],
        json=payload,
        timeout=600
    )

    latency = time.time() - start

    completion_tokens = 0

    try:
        data = r.json()
        completion_tokens = data.get("usage", {}).get("completion_tokens", 0)
    except Exception:
        pass

    return {
        "status": r.status_code,
        "latency": latency,
        "tokens": completion_tokens
    }


In [ ]:
def benchmark(target, prompt, users, requests_total):

    start = time.time()
    results = []

    with ThreadPoolExecutor(max_workers=users) as pool:

        futures = [
            pool.submit(send_request, target, prompt)
            for _ in range(requests_total)
        ]

        for f in as_completed(futures):
            results.append(f.result())

    duration = time.time() - start

    ok = [r for r in results if r["status"] == 200]

    latencies = [r["latency"] for r in ok]

    return {
        "users": users,
        "success": len(ok),
        "errors": requests_total - len(ok),
        "avg_latency": statistics.mean(latencies) if latencies else None,
        "p95_latency": sorted(latencies)[int(0.95 * len(latencies))] if latencies else None,
        "token_per_sec": sum(r["tokens"] for r in ok) / duration if duration else 0,
        "req_per_sec": requests_total / duration
    }


In [ ]:
CONCURRENT_USERS = [1, 10, 25, 50, 100, 200]
REQUESTS_PER_TEST = 100


In [ ]:
def run_suite(target, prompt):

    rows = []

    for users in CONCURRENT_USERS:
        print(f"{target} users={users}")
        rows.append(
            benchmark(
                target,
                prompt,
                users,
                REQUESTS_PER_TEST
            )
        )

    return pd.DataFrame(rows)


In [ ]:
# Short prompt benchmark

vllm_short = run_suite("vllm", SHORT_PROMPT)
llmd_short = run_suite("llmd", SHORT_PROMPT)


In [ ]:
# Long prompt benchmark

vllm_long = run_suite("vllm", LONG_PROMPT)
llmd_long = run_suite("llmd", LONG_PROMPT)


In [ ]:
comparison = vllm_long.merge(
    llmd_long,
    on="users",
    suffixes=("_vllm", "_llmd")
)

comparison


In [ ]:
plt.figure(figsize=(8,5))

plt.plot(comparison["users"], comparison["token_per_sec_vllm"], marker="o", label="vLLM")
plt.plot(comparison["users"], comparison["token_per_sec_llmd"], marker="o", label="llm-d")

plt.xlabel("Concurrent Users")
plt.ylabel("Token/sec")
plt.legend()
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))

plt.plot(comparison["users"], comparison["p95_latency_vllm"], marker="o", label="vLLM")
plt.plot(comparison["users"], comparison["p95_latency_llmd"], marker="o", label="llm-d")

plt.xlabel("Concurrent Users")
plt.ylabel("P95 Latency (s)")
plt.legend()
plt.grid()
plt.show()
